# Memory Allocator Mock Interview

This allocator exercise can be copied into CoderPad or run directly in
Colab/Jupyter. It focuses on first-fit allocation, free-block coalescing,
fragmentation, and improving an `O(n)` linked-list baseline to expected
`O(log m)` operations.

## Running the exercise

```bash
cd algorithm-coding-challenge/memory-allocator-puzzle

# Single-file candidate mock; initially fails at the first TODO.
python3 memory_allocator_mock.py

# Augmented-treap reference implementation.
python3 solution.py

# Generate and validate the Colab/Jupyter version.
python3 scripts/generate_notebook.py
python3 scripts/check_notebook.py
```

`memory_allocator_mock.py` contains the function signatures, fixed tests, a
bitmap slow oracle, 80,000 deterministic random operations, a
fragmentation/scaling test, and `run_all_tests()`. It uses only the Python
standard library.

## Part 1: Pointer-only API

```python
class Allocator:
    def __init__(self, size: int = 1000) -> None: ...
    def malloc(self, size: int) -> int: ...
    def free(self, pointer: int) -> bool: ...
```

- The address space is the half-open interval `[0, size)` and starts entirely
  free.
- `malloc(size)` uses **first-fit by address**: select the lowest-address free
  gap large enough for the request and allocate from its left edge.
- Return `-1` when no contiguous gap can satisfy the request.
- Raise `ValueError` when `size <= 0`.
- `free(pointer)` releases one complete allocation and automatically merges
  adjacent free gaps.
- Return `False` for an unknown pointer or double-free.

Introspection helpers:

```python
def get_free_memory(self) -> int: ...
def get_largest_free_block(self) -> int: ...
```

These expose external fragmentation: sufficient total free memory does not
guarantee that a sufficiently large contiguous block exists.

## Part 2: Explicit-size API

```python
class MemoryAllocator:
    def __init__(self, total_capacity: int) -> None: ...
    def allocate(self, size: int) -> int: ...
    def free(self, address: int, size: int) -> None: ...
```

The allocation strategy matches Part 1, with these contract changes:

- `allocate` raises `MemoryError` when no contiguous block is large enough.
- The caller must pass the exact `(address, size)` pair back to `free`.
- A non-positive size, out-of-range interval, unknown address, incorrect size,
  or double-free raises `ValueError`.
- A `{address: size}` live-allocation map validates every free request.

Example:

```python
alloc = MemoryAllocator(100)
a = alloc.allocate(20)  # 0
b = alloc.allocate(30)  # 20
c = alloc.allocate(40)  # 50
alloc.free(b, 30)
d = alloc.allocate(25)  # 20; leaves free [45, 50)
alloc.free(a, 20)
alloc.free(d, 25)       # merges into free [0, 50)
```

## The four free/merge cases

When freeing `[start, end)`, locate its predecessor and successor by address:

| Left adjacent | Right adjacent | Operation |
| --- | --- | --- |
| No | No | Insert a new gap |
| Yes | No | Extend the left gap |
| No | Yes | Merge the freed block with the right gap |
| Yes | Yes | Collapse all three intervals into one gap |

Only adjacent intervals may be merged. A live allocation can never be skipped.

## Complexity progression

### Baseline: sorted doubly linked list

Store free gaps in start-address order using nodes containing
`{start, size, prev, next}`:

- `allocate` scans linearly for first fit: `O(m)`.
- `free` scans linearly for the insertion point and then merges neighbors in
  `O(1)`, for `O(m)` overall.
- Space is `O(m)`.

This is a reasonable first implementation but does not satisfy the performance
follow-up.

### Reference: augmented treap

`solution.py` uses a randomized treap keyed by gap start address. Every
subtree stores:

- `max_size`: its largest gap, allowing the search to skip any subtree that
  cannot satisfy the request.
- `total_size`: its total number of free bytes.

`_leftmost_fit` checks the left subtree, then the current node, then the right
subtree. This preserves **leftmost first-fit**, not best-fit. Allocation,
predecessor/successor lookup, insertion, erasure, and coalescing take expected
`O(log m)`. The two statistics are available in `O(1)`, and space is `O(m)`.

For deterministic worst-case `O(log m)`, replace the treap with an AVL or
red-black tree while retaining the same subtree augmentation.

## Follow-ups

| Topic | Discussion direction |
| --- | --- |
| Alignment | Round requests up and account for padding before an aligned gap start |
| Best-fit | Maintain a second `(size, start)` index and choose the smallest sufficient gap |
| Realloc | Grow into the right gap when possible; otherwise allocate, copy, and free |
| Thread safety | Begin with a global lock, then discuss size-class or arena sharding |
| Fragmentation | Segregated free lists, buddy allocation, and compaction |
| Metadata overhead | Implicit free lists, in-memory headers, and boundary tags |
| Fixed-size blocks | Bitmap or slab allocation |
| Invalid access | Generation IDs, guard regions, and poisoning; an allocator alone cannot fully prevent use-after-free |

A buddy system uses power-of-two blocks to make splitting and coalescing fast,
at the cost of internal fragmentation. Boundary tags place metadata at block
boundaries so an allocator can locate the previous block when metadata lives
inside the managed memory.

## Test coverage

- Exact fit, full memory, and capacity boundaries.
- Leftmost gap reuse and external fragmentation.
- No-, left-, right-, and both-neighbor merge cases.
- Invalid size, unknown pointer, incorrect-size free, and double-free.
- Random differential testing against the bitmap slow oracle.
- Search and coalescing after creating many fragmented one-byte blocks.


In [ ]:
"""CoderPad-style memory allocator interview mock.

Implement both allocator APIs, then run this file. The fixed and deterministic
random tests compare behavior with a deliberately slow bitmap oracle.
"""

from __future__ import annotations

import random
import sys


class Allocator:
    """Pointer-only API. Target: expected O(log m) per operation."""

    def __init__(self, size: int = 1000) -> None:
        raise NotImplementedError

    def malloc(self, size: int) -> int:
        """Allocate from the leftmost sufficient gap; return -1 on exhaustion."""

        raise NotImplementedError

    def free(self, pointer: int) -> bool:
        """Free a live allocation; return False for invalid or double free."""

        raise NotImplementedError

    def get_free_memory(self) -> int:
        raise NotImplementedError

    def get_largest_free_block(self) -> int:
        raise NotImplementedError


class MemoryAllocator:
    """Explicit-size API. Target: expected O(log m) per operation."""

    def __init__(self, total_capacity: int) -> None:
        raise NotImplementedError

    def allocate(self, size: int) -> int:
        """Allocate first-fit; raise MemoryError when no contiguous gap fits."""

        raise NotImplementedError

    def free(self, address: int, size: int) -> None:
        """Free exactly one recorded allocation, validating address and size."""

        raise NotImplementedError

    def get_free_memory(self) -> int:
        raise NotImplementedError

    def get_largest_free_block(self) -> int:
        raise NotImplementedError


class _BitmapAllocatorOracle:
    def __init__(self, capacity: int = 1000) -> None:
        if capacity <= 0:
            raise ValueError("capacity must be positive")
        self.capacity = capacity
        self.free_bytes = [True] * capacity
        self.allocations: dict[int, int] = {}

    def malloc(self, size: int) -> int:
        if size <= 0:
            raise ValueError("size must be positive")
        run = 0
        for index, is_free in enumerate(self.free_bytes):
            run = run + 1 if is_free else 0
            if run == size:
                start = index - size + 1
                self.free_bytes[start : index + 1] = [False] * size
                self.allocations[start] = size
                return start
        return -1

    def free(self, pointer: int) -> bool:
        size = self.allocations.pop(pointer, None)
        if size is None:
            return False
        self.free_bytes[pointer : pointer + size] = [True] * size
        return True

    def get_free_memory(self) -> int:
        return sum(self.free_bytes)

    def get_largest_free_block(self) -> int:
        best = run = 0
        for is_free in self.free_bytes:
            run = run + 1 if is_free else 0
            best = max(best, run)
        return best


class _ExplicitBitmapOracle(_BitmapAllocatorOracle):
    def allocate(self, size: int) -> int:
        address = self.malloc(size)
        if address == -1:
            raise MemoryError("no contiguous free block is large enough")
        return address

    def free_explicit(self, address: int, size: int) -> None:
        if size <= 0:
            raise ValueError("size must be positive")
        if address < 0 or address >= self.capacity:
            raise ValueError("address is outside the allocator")
        if address + size > self.capacity:
            raise ValueError("range exceeds allocator capacity")
        recorded = self.allocations.get(address)
        if recorded is None:
            raise ValueError("address is not a live allocation")
        if recorded != size:
            raise ValueError("size does not match the live allocation")
        super().free(address)


def _expect_equal(actual, expected, label: str) -> None:
    if actual != expected:
        raise AssertionError(f"{label}: got {actual!r}, expected {expected!r}")


def _expect_raises(error_type, fn, *args) -> None:
    try:
        fn(*args)
    except error_type:
        return
    except Exception as error:
        raise AssertionError(
            f"{fn.__name__} raised {type(error).__name__}, "
            f"expected {error_type.__name__}"
        ) from error
    raise AssertionError(f"{fn.__name__}{args!r} should raise {error_type.__name__}")


def _assert_stats(allocator, free_memory: int, largest: int, label: str) -> None:
    _expect_equal(allocator.get_free_memory(), free_memory, f"{label} free")
    _expect_equal(
        allocator.get_largest_free_block(), largest, f"{label} largest"
    )


def _fixed_pointer_tests(impl) -> None:
    default = impl.Allocator()
    _assert_stats(default, 1000, 1000, "default capacity")

    allocator = impl.Allocator(100)
    a = allocator.malloc(20)
    b = allocator.malloc(30)
    c = allocator.malloc(40)
    _expect_equal((a, b, c), (0, 20, 50), "sequential allocation")
    _assert_stats(allocator, 10, 10, "after three allocations")

    _expect_equal(allocator.free(b), True, "free middle")
    d = allocator.malloc(25)
    _expect_equal(d, 20, "reuse leftmost middle gap")
    _assert_stats(allocator, 15, 10, "fragmented state")

    _expect_equal(allocator.free(a), True, "free left")
    _expect_equal(allocator.free(d), True, "merge both sides")
    _assert_stats(allocator, 60, 50, "both-neighbor merge")
    _expect_equal(allocator.free(d), False, "double free")
    _expect_equal(allocator.free(999), False, "unknown pointer")

    exact = impl.Allocator(8)
    _expect_equal(exact.malloc(8), 0, "exact fit")
    _expect_equal(exact.malloc(1), -1, "full allocator")
    _assert_stats(exact, 0, 0, "full state")
    _expect_equal(exact.free(0), True, "free exact fit")
    _assert_stats(exact, 8, 8, "fully coalesced")

    fragmented = impl.Allocator(12)
    pointers = [fragmented.malloc(3) for _ in range(4)]
    fragmented.free(pointers[0])
    fragmented.free(pointers[2])
    _expect_equal(fragmented.malloc(4), -1, "external fragmentation")
    _expect_equal(fragmented.malloc(3), 0, "leftmost first fit")

    merges = impl.Allocator(25)
    blocks = [merges.malloc(5) for _ in range(5)]
    _expect_equal(merges.free(blocks[1]), True, "no-neighbor merge case")
    _expect_equal(merges.free(blocks[3]), True, "second isolated gap")
    _expect_equal(merges.free(blocks[2]), True, "both-neighbor merge case")
    _assert_stats(merges, 15, 15, "after both-neighbor merge")
    _expect_equal(merges.free(blocks[0]), True, "right-neighbor merge case")
    _expect_equal(merges.free(blocks[4]), True, "left-neighbor merge case")
    _assert_stats(merges, 25, 25, "all four merge cases")

    _expect_raises(ValueError, impl.Allocator, 0)
    _expect_raises(ValueError, impl.Allocator, -1)
    _expect_raises(ValueError, allocator.malloc, 0)
    _expect_raises(ValueError, allocator.malloc, -3)


def _fixed_explicit_tests(impl) -> None:
    allocator = impl.MemoryAllocator(100)
    a = allocator.allocate(20)
    b = allocator.allocate(30)
    c = allocator.allocate(40)
    _expect_equal((a, b, c), (0, 20, 50), "explicit sequence")
    allocator.free(20, 30)
    d = allocator.allocate(25)
    _expect_equal(d, 20, "explicit gap reuse")
    allocator.free(0, 20)
    allocator.free(20, 25)
    _assert_stats(allocator, 60, 50, "explicit both-side merge")

    _expect_raises(ValueError, impl.MemoryAllocator, 0)
    _expect_raises(ValueError, allocator.allocate, 0)
    _expect_raises(MemoryError, allocator.allocate, 51)
    _expect_raises(ValueError, allocator.free, -1, 1)
    _expect_raises(ValueError, allocator.free, 100, 1)
    _expect_raises(ValueError, allocator.free, 90, 11)
    _expect_raises(ValueError, allocator.free, 50, 39)
    allocator.free(50, 40)
    _expect_raises(ValueError, allocator.free, 50, 40)
    _assert_stats(allocator, 100, 100, "explicit full merge")


def _compare_stats(actual, expected, label: str) -> None:
    _expect_equal(
        actual.get_free_memory(),
        expected.get_free_memory(),
        f"{label} free memory",
    )
    _expect_equal(
        actual.get_largest_free_block(),
        expected.get_largest_free_block(),
        f"{label} largest gap",
    )


def _random_pointer_tests(impl, rng: random.Random) -> None:
    for case in range(200):
        capacity = rng.randint(1, 80)
        actual = impl.Allocator(capacity)
        expected = _BitmapAllocatorOracle(capacity)
        known_pointers: list[int] = []
        for step in range(200):
            if rng.random() < 0.62:
                size = rng.randint(1, capacity + 10)
                actual_pointer = actual.malloc(size)
                expected_pointer = expected.malloc(size)
                _expect_equal(
                    actual_pointer,
                    expected_pointer,
                    f"pointer random case {case}, step {step}",
                )
                if actual_pointer != -1:
                    known_pointers.append(actual_pointer)
            else:
                if known_pointers and rng.random() < 0.8:
                    pointer = rng.choice(known_pointers)
                else:
                    pointer = rng.randint(-2, capacity + 2)
                _expect_equal(
                    actual.free(pointer),
                    expected.free(pointer),
                    f"pointer free case {case}, step {step}",
                )
            _compare_stats(actual, expected, f"pointer case {case}, step {step}")


def _random_explicit_tests(impl, rng: random.Random) -> None:
    for case in range(200):
        capacity = rng.randint(1, 80)
        actual = impl.MemoryAllocator(capacity)
        expected = _ExplicitBitmapOracle(capacity)
        live: dict[int, int] = {}
        for step in range(200):
            if rng.random() < 0.62:
                size = rng.randint(1, capacity + 10)
                try:
                    expected_address = expected.allocate(size)
                except MemoryError:
                    _expect_raises(MemoryError, actual.allocate, size)
                else:
                    actual_address = actual.allocate(size)
                    _expect_equal(
                        actual_address,
                        expected_address,
                        f"explicit allocate case {case}, step {step}",
                    )
                    live[actual_address] = size
            elif live and rng.random() < 0.85:
                address = rng.choice(list(live))
                size = live.pop(address)
                actual.free(address, size)
                expected.free_explicit(address, size)
            else:
                if live and rng.random() < 0.5:
                    address = rng.choice(list(live))
                    size = live[address] + 1
                else:
                    address = capacity
                    size = 1
                _expect_raises(ValueError, actual.free, address, size)
                _expect_raises(ValueError, expected.free_explicit, address, size)
            _compare_stats(actual, expected, f"explicit case {case}, step {step}")


def _scaling_smoke_test(impl) -> None:
    allocator = impl.Allocator(25_000)
    pointers = [allocator.malloc(1) for _ in range(25_000)]
    for pointer in pointers[::2]:
        if not allocator.free(pointer):
            raise AssertionError("scaling test free failed")
    for expected in range(0, 25_000, 2):
        _expect_equal(
            allocator.malloc(1), expected, "scaling test leftmost fit"
        )
    _assert_stats(allocator, 0, 0, "scaling test final")


def _run_all_tests(impl) -> None:
    _fixed_pointer_tests(impl)
    _fixed_explicit_tests(impl)
    print("PASS fixed tests")
    rng = random.Random(20260915)
    _random_pointer_tests(impl, rng)
    _random_explicit_tests(impl, rng)
    print("PASS 80,000 deterministic random operations")
    _scaling_smoke_test(impl)
    print("PASS 50,000-operation fragmentation smoke test")


def run_all_tests() -> None:
    _run_all_tests(sys.modules[__name__])

In [ ]:
run_all_tests()
